# LC 739 — Daily Temperatures
**Day 45 | Theme: Monotonic Stack | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Keep a <em>monotonic decreasing</em> stack
of indices. The moment a warmer day arrives, every index on the stack
that is cooler gets its answer resolved in one pass — no nested loops
needed.
</div>

## Official Problem Statement

Given an array of integers `temperatures` representing daily temperatures,
return an array `answer` such that `answer[i]` is the number of days you
have to wait after the `i`-th day to get a warmer temperature. If there
is no future day with a warmer temperature, keep `answer[i] == 0`.

**Constraints:**
- `1 <= temperatures.length <= 10^5`
- `30 <= temperatures[i] <= 100`

## What This Is Actually Asking

For each day, find the *next* day that is strictly warmer.
The answer is the gap (in days) between them, not the temperature
itself.
If no warmer day exists, the gap is 0.
Brute force checks every future day for each day — O(n²).
A monotonic stack lets us resolve multiple days at once when a new
peak arrives, achieving O(n).

## Walk Through an Example by Hand

```
temperatures = [73, 74, 75, 71, 69, 72, 76, 73]
indices        [ 0,  1,  2,  3,  4,  5,  6,  7]
```

| Step | i | temp | Stack before | Action | Stack after |
|------|---|------|-------------|--------|-------------|
| 1    | 0 |  73  | []          | push 0 | [0]         |
| 2    | 1 |  74  | [0]         | 74>73→pop0,ans[0]=1; push1 | [1] |
| 3    | 2 |  75  | [1]         | 75>74→pop1,ans[1]=1; push2 | [2] |
| 4    | 3 |  71  | [2]         | 71<75→push3 | [2,3]    |
| 5    | 4 |  69  | [2,3]       | 69<71→push4 | [2,3,4]  |
| 6    | 5 |  72  | [2,3,4]     | 72>69→pop4,ans[4]=1; 72>71→pop3,ans[3]=2; 72<75→push5 | [2,5] |
| 7    | 6 |  76  | [2,5]       | 76>72→pop5,ans[5]=1; 76>75→pop2,ans[2]=4; push6 | [6] |
| 8    | 7 |  73  | [6]         | 73<76→push7 | [6,7]    |

End: indices 6 and 7 remain → ans[6]=0, ans[7]=0

**Result:** `[1, 1, 4, 2, 1, 1, 0, 0]`

## The Picture

```
Temp
 76 |                 █
 75 |       █         █
 74 |   █   █         █
 73 | █ █   █         █ █
 72 |   █   █   █     █ █
 71 |   █   █ █ █     █ █
 69 |   █   █ █ █ █   █ █
     -----------------------
      0  1  2  3  4  5  6  7

Stack state (indices) at each push:
  After i=0: [0]          ← only 73, nothing warmer yet
  After i=2: [2]          ← 75 cleared 74 and 73
  After i=4: [2, 3, 4]   ← decreasing temps piling up
  After i=5: [2, 5]      ← 72 cleared 69 and 71
  After i=6: [6]         ← 76 cleared everything
  Final:     [6, 7]      ← no warmer day → answer 0

Arrow shows "next warmer day":
  0→1  1→2  2→6  3→5  4→5  5→6  6=0  7=0
```

**Key visual:** The stack always holds a *descending staircase* of
temperatures. A new high clears the staircase from the top.

## When To Use This Pattern

- When you need the **next greater element** to the right, think
  monotonic decreasing stack.
- When each element's answer depends on a **future element not yet
  seen**, think stack + deferred resolution.
- When brute force is O(n²) "next bigger" scans, think stack O(n).
- When the problem says "how many days / steps until X exceeds Y",
  think index gap from a stack pop.
- When elements can **resolve multiple earlier pending elements** at
  once, think monotonic stack batch pop.

## The Approach

Maintain a stack of indices whose temperatures are still waiting for a
warmer day. Iterate left to right; for each index `i`, pop every index
from the stack whose temperature is less than `temperatures[i]` — those
days just found their answer as `i - popped_index`. Push the current
index onto the stack. Any indices remaining in the stack at the end
have no warmer day and keep the default answer of 0.

In [10]:
from typing import List

In [11]:
def test_harness(func):
    """Run test cases for Daily Temperatures."""
    cases = [
        # (temperatures, expected)
        ([73, 74, 75, 71, 69, 72, 76, 73],
         [1, 1, 4, 2, 1, 1, 0, 0]),
        ([30, 40, 50, 60],
         [1, 1, 1, 0]),
        ([30, 60, 90],
         [1, 1, 0]),
        # Edge: single element
        ([55],
         [0]),
        # Edge: all same temperature
        ([70, 70, 70],
         [0, 0, 0]),
        # Edge: strictly decreasing
        ([90, 80, 70, 60],
         [0, 0, 0, 0]),
    ]
    passed = 0
    for i, (temps, expected) in enumerate(cases):
        result = func(temps)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"  Case {i}: {status}")
            print(f"    Input:    {temps}")
            print(f"    Expected: {expected}")
            print(f"    Got:      {result}")
    total = len(cases)
    print(f"\n  Summary: {passed}/{total} passed")
    if passed == total:
        print("  All tests PASSED!")

In [12]:
def daily_temperatures(temps: List[int]) -> List[int]:
    """
    Return wait days until next warmer temperature.

    Strategy: Monotonic decreasing stack of indices.
    - Iterate through temperatures left to right.
    - While top of stack has a cooler temp than current,
      pop and record the gap.
    - Push current index.

    Args:
        temperatures: List of daily temperatures (30-100).

    Returns:
        List where result[i] = days until warmer day
        (0 if none exists).

    Time:  O(n) — each index pushed/popped at most once.
    Space: O(n) — stack and result array.
    """
    # Initialize a zero list for every day.
    # Initialize an empty stack to house indices of monotonic decreasing temperatures.
    # For a temperature warmer than the stack top, evict the index and populate the result.
    # Example: daily_temperatures([90, 60, 30]) -> [0, 0, 0] (a decreasing monotonic stack).
    stack = []
    res = [0] * len(temps)
    for i, temp in enumerate(temps):
        while stack and temp > temps[stack[-1]]:
            idx = stack.pop()
            res[idx] = i - idx
        stack.append(i)
    return res
"""
[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0]
[0, 0, 0]
[0]

  Summary: 6/6 passed
  All tests PASSED!
"""
 # Main example — expect [1,1,4,2,1,1,0,0]
print(daily_temperatures([73,74,75,71,69,72,76,73]))

# Strictly increasing — expect [1,1,1,0]
print(daily_temperatures([30,40,50,60]))

# Strictly decreasing — expect [0,0,0]
print(daily_temperatures([90,60,30]))

# All same — expect [0,0,0]
print(daily_temperatures([70,70,70]))

# Single element — expect [0]
print(daily_temperatures([55]))

r'''
def test_harness(func):
    """Run test cases for Daily Temperatures."""
    cases = [
        # (temperatures, expected)
        ([73, 74, 75, 71, 69, 72, 76, 73],
         [1, 1, 4, 2, 1, 1, 0, 0]),
        ([30, 40, 50, 60],
         [1, 1, 1, 0]),
        ([30, 60, 90],
         [1, 1, 0]),
        # Edge: single element
        ([55],
         [0]),
        # Edge: all same temperature
        ([70, 70, 70],
         [0, 0, 0]),
        # Edge: strictly decreasing
        ([90, 80, 70, 60],
         [0, 0, 0, 0]),
    ]
    passed = 0
    for i, (temps, expected) in enumerate(cases):
        result = func(temps)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"  Case {i}: {status}")
            print(f"    Input:    {temps}")
            print(f"    Expected: {expected}")
            print(f"    Got:      {result}")
    total = len(cases)
    print(f"\n  Summary: {passed}/{total} passed")
    if passed == total:
        print("  All tests PASSED!")
'''
test_harness(daily_temperatures)   

[1, 1, 4, 2, 1, 1, 0, 0]
[1, 1, 1, 0]
[0, 0, 0]
[0, 0, 0]
[0]

  Summary: 6/6 passed
  All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# test_harness(daily_temperatures)

## Complexity

| Approach      | Time   | Space  | Notes                          |
|---------------|--------|--------|--------------------------------|
| Brute Force   | O(n²)  | O(1)   | Nested loop for each day       |
| Monotonic Stack | O(n) | O(n)   | Each index pushed/popped once  |

## Real World Connection

At **Citi**, risk engines scan time-series market data to find the next
day a metric (e.g., bond yield) exceeds today's reading — exactly the
"next warmer day" pattern. On **AWS**, CloudWatch alarm evaluation uses
a similar deferred-resolution concept: pending breaches wait until a
threshold is crossed to fire. In **data engineering**, streaming
pipelines often need to tag each record with the timestamp of the next
event that surpasses a threshold; a monotonic stack on sorted event
streams achieves this in a single O(n) pass rather than repeated
window scans.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra